In [1]:
import pandas as pd
import tensorflow as tf
import numpy as np
import copy
import math

In [2]:
batch_size = 256
l_rate = 0.001

In [3]:
@tf.keras.utils.register_keras_serializable()
class MLP(tf.keras.Model):
    def __init__(self, trainable=True, dtype=None, **kwargs):
        super().__init__(trainable=trainable, dtype=dtype, **kwargs)
        self.dense1 = tf.keras.layers.Dense(units=512, activation=tf.nn.leaky_relu)
        self.dense2 = tf.keras.layers.Dense(units=1024, activation=tf.nn.leaky_relu)
        self.dense3 = tf.keras.layers.Dense(units=512, activation=tf.nn.leaky_relu)
        self.dense4 = tf.keras.layers.Dense(units=256, activation=tf.nn.leaky_relu)
        self.dense5 = tf.keras.layers.Dense(units=8)

    def get_config(self):
        config = super().get_config()
        return config

    @classmethod
    def from_config(cls, config):
        return cls(**config)

    def call(self, inputs):
        x = self.dense1(inputs)
        x = self.dense2(x)
        x = self.dense3(x)
        x = self.dense4(x)
        output = self.dense5(x)
        return output

In [4]:
class ParaServer:
    def __init__(self):
        self.model = MLP()
        self.sandbox_model = MLP()
        self.optimizer = tf.keras.optimizers.Adam(learning_rate=l_rate)
        self.sandbox_opt = tf.keras.optimizers.Adam(learning_rate=l_rate)
        self.freqs = {}
        test_dataset = pd.read_csv("Test.csv", encoding='utf-8').sample(frac=1).reset_index(drop=True)
        self.X_v = test_dataset.loc[:,'freq':'L4'].to_numpy(dtype = np.float32)
        self.y_v = test_dataset.loc[:,'S11r':'S41i'].to_numpy(dtype = np.float32)
        self.r2sv = []
        self.model(self.X_v)
        self.sandbox_model(self.X_v)
        self.D = 0
        self.A = 0

    def upload(self, grads, freq, score):
        origin_error = self.testset_vali(self.model)
        self.sandbox_model.set_weights(self.model.get_weights())
        self.sandbox_opt.apply_gradients(grads_and_vars=zip(grads, self.sandbox_model.variables))
        sandbox_error = self.testset_vali(self.sandbox_model)
        
        if sandbox_error > origin_error:
            print("D", end="")
            self.D += 1
        else:
            print("A", end="")
            self.A += 1
        
        self.freqs[freq] = max(0, score)
        self.optimizer.apply_gradients(grads_and_vars=zip(grads, self.model.variables))
        return self.model, self.freqs
    
    def download(self):
        return self.model, self.freqs

    def lr_decay(self, ratio):
        self.optimizer.learning_rate = self.optimizer.learning_rate * ratio

    def testset_vali(self, model, show=False):
        y_v_p = model(self.X_v)
        va_mse = tf.reduce_mean(tf.square(y_v_p - self.y_v))
        va_rmse = tf.sqrt(va_mse)
        va_mae = tf.reduce_mean(tf.abs(y_v_p - self.y_v))
        va_r2 = 1 - tf.reduce_sum(tf.square(y_v_p - self.y_v)) / tf.reduce_sum(tf.square(self.y_v - tf.reduce_mean(self.y_v)))
        if show:
            print("vali - mse:{} rmse:{} mae:{} r2:{}".format(va_mse, va_rmse, va_mae, va_r2))
            self.r2sv.append(va_r2.numpy())
        return va_mse

In [5]:
r2s = {2.4:[],2.5:[],2.6:[]}

In [6]:
ps = ParaServer()

In [7]:
class Node:
    def __init__(self, dsName, freq):
        self.freq = freq
        self.otfreqs = {}
        self.model = MLP()
        self.zeroModel = MLP()
        self.dataset = pd.read_csv(dsName, encoding='utf-8').sample(frac=1).reset_index(drop=True)
        self.X = self.dataset.loc[:,'freq':'L4'].to_numpy(dtype = np.float32)
        self.y = self.dataset.loc[:,'S11r':'S41i'].to_numpy(dtype = np.float32)
        self.dataset_train = tf.data.Dataset.from_tensor_slices((self.X, self.y))
        self.dataset_train = self.dataset_train.shuffle(buffer_size=self.X.shape[0])
        self.dataset_train = self.dataset_train.batch(batch_size)
        self.dataset_train = self.dataset_train.prefetch(tf.data.experimental.AUTOTUNE)
    def getZero(self):
        m, freqs = ps.download()
        self.otfreqs = copy.deepcopy(freqs)
        self.zeroModel = copy.deepcopy(m)
        print(self.otfreqs)
    def train(self, index_epoch):
        self.model, _ = ps.download()
        for X, y in self.dataset_train:
            with tf.GradientTape() as tape:
                y_pred = self.model(X)
                tr_mse = tf.reduce_mean(tf.square(y_pred - y))
            tr_rmse = tf.sqrt(tr_mse)
            tr_mae = tf.reduce_mean(tf.abs(y_pred - y))
            tr_r2 = 1 - tf.reduce_sum(tf.square(y_pred - y)) / tf.reduce_sum(tf.square(y - tf.reduce_mean(y)))
            grads = tape.gradient(tr_mse, self.model.variables)
            sum_r2 = 1
            for k, v in self.otfreqs.items():
                if math.isclose(k, self.freq) or math.isclose(v, 0):
                    continue
                X_i = tf.tensor_scatter_nd_update(X, [[i, 0] for i in range(X.shape[0])], [k] * X.shape[0])
                y_i = self.zeroModel(X_i)
                with tf.GradientTape() as tape:
                    y_pred_i = self.model(X_i)
                    loss = tf.reduce_mean(tf.square(y_pred_i - y_i))
                grad = tape.gradient(loss, self.model.variables)
                grads = [grads[i] + grad[i] * v for i in range(len(grads))]
                sum_r2 += v
            self.model, _ = ps.upload([i / sum_r2 for i in grads], self.freq, tr_r2.numpy())
        # if epoch_index in np.arange(0, num_epochs, 25).tolist() or epoch_index == num_epochs - 1:
        print(f"({ps.A}/{ps.D})")
        ps.A = 0
        ps.D = 0
        print("node:{} epoch:{}".format(self.freq, index_epoch))
        print("train - mse:{} rmse:{} mae:{} r2:{}".format(tr_mse, tr_rmse, tr_mae, tr_r2))
        r2s[self.freq].append(tr_r2.numpy())
        # ps.testset_vali(show=True)

In [8]:
nodeList = [Node('./24Train.csv', 2.4), Node('./25Train.csv', 2.5), Node('./26Train.csv', 2.6)]

In [9]:
for i in range(2):
    for j in range(3):
        nodeList[j].getZero()
        for k in range(100):
            nodeList[j].train(k)
    ps.lr_decay(0.1)


{}
DAADAAADAAADDDAAADDDAAADDDAAAADDDAAAADDDAAAADDDAAAADDDAAAADDAAADADDADADADADADADADAAADAAADADADADADADADDADADAADADD(62/50)
node:2.4 epoch:0
train - mse:0.08812185376882553 rmse:0.296853244304657 mae:0.24138160049915314 r2:0.2749199867248535
ADADADADDDAAADDADAADDADADADADAADDADADDADAADADADADDADADDADAAADAADDADADADADAADADADADAADADDADAADADADAADDADDADDADADD(55/57)
node:2.4 epoch:1
train - mse:0.08328980207443237 rmse:0.28859972953796387 mae:0.23292477428913116 r2:0.30446386337280273
AADADDADADDADDADDADADAADADDADADDADADAADADADDADAADADDADAADADDADAADAADDADADADAADDADADDAADADAADDADAADADAAAADAADADAD(57/55)
node:2.4 epoch:2
train - mse:0.07655879110097885 rmse:0.27669259905815125 mae:0.22404107451438904 r2:0.36680537462234497
ADDADADDADADAAADADAADDDAAADADADADADADDADADADADADAADDADAADAADAADDADADAADADADADADAADDADADADAADAADADDAAADAADADDADAA(60/52)
node:2.4 epoch:3
train - mse:0.07874363660812378 rmse:0.280612975358963 mae:0.22739726305007935 r2:0.34643441438674927
DADADADADAADAADADADDADDADADADADDADADA

KeyboardInterrupt: 